S&P 100 Historical Composition Tracker
========================================
Scrapes weekly snapshots of the Wikipedia S&P 100 article between
two dates, extracts the components table from each revision, and
classifies companies as CONSTANT or TRANSIENT over the study period.

Requirements:
    pip install requests beautifulsoup4 pandas tqdm

Usage:
    python sp100_composition_tracker.py

    By default covers 2021-01-01 to 2025-12-31.
    Edit START_DATE / END_DATE / SAMPLE_FREQ_DAYS at the top to change.

Outputs (written to ./output/):
    - revisions_index.csv      : All revision IDs sampled with timestamps
    - weekly_composition.csv   : Long-format table (date, ticker, company)
    - composition_matrix.csv   : Wide-format pivot (rows=weeks, cols=tickers, 1/0)
    - constant_companies.csv   : Tickers present in EVERY sampled week
    - transient_companies.csv  : Tickers absent in at least one week (with stats)
    - summary_report.txt       : Human-readable summary

In [11]:
import time
import datetime
import os
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

START_DATE       = datetime.date(2021, 1, 1)
END_DATE         = datetime.date(2025, 12, 31)
SAMPLE_FREQ_DAYS = 7          # one snapshot per week
SLEEP_BETWEEN_REQUESTS = 1.2  # seconds — be polite to Wikipedia's servers
OUTPUT_DIR       = "./sp100_output"

WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_OLD = "https://en.wikipedia.org/w/index.php"
ARTICLE  = "S&P_100"

HEADERS = {
    "User-Agent": (
        "SP100CompositionTracker/1.0 "
        "(academic research; https://github.com/Javirios03/sp100-graph-forecasting)"
        "python-requests/2.x"
    )
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

In [12]:
def get_revision_ids(start: datetime.date, end: datetime.date) -> list[dict]:
    """
    Fetch all revision IDs for the S&P_100 article between start and end.
    Returns a list of dicts: {revid, timestamp}
    """
    revisions = []
    params = {
        "action":    "query",
        "titles":    ARTICLE,
        "prop":      "revisions",
        "rvprop":    "ids|timestamp",
        "rvlimit":   "500",
        "rvstart":   f"{start}T00:00:00Z",
        "rvend":     f"{end}T23:59:59Z",
        "rvdir":     "newer",
        "format":    "json",
    }
    while True:
        resp = SESSION.get(WIKI_API, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        pages = data["query"]["pages"]
        page  = next(iter(pages.values()))
        for rev in page.get("revisions", []):
            revisions.append({
                "revid":     rev["revid"],
                "timestamp": rev["timestamp"],
            })

        if "continue" in data:
            params["rvcontinue"] = data["continue"]["rvcontinue"]
            time.sleep(SLEEP_BETWEEN_REQUESTS)
        else:
            break

    return revisions

In [13]:
def sample_weekly_revisions(revisions: list[dict],
                             start: datetime.date,
                             end: datetime.date,
                             freq_days: int = 7) -> list[dict]:
    """
    Given a full list of revisions, pick the LAST revision available
    for each weekly checkpoint from start to end.
    This ensures we always have a revision ≤ the checkpoint date.
    """
    # Build a lookup: date → last revid on or before that date
    # Convert timestamps to dates
    df = pd.DataFrame(revisions)
    df["date"] = pd.to_datetime(df["timestamp"]).dt.date
    df = df.sort_values("date")

    checkpoints = []
    current = start
    while current <= end:
        # Last revision on or before current checkpoint
        subset = df[df["date"] <= current]
        if not subset.empty:
            row = subset.iloc[-1]
            checkpoints.append({
                "checkpoint": current,
                "revid":      row["revid"],
                "timestamp":  row["timestamp"],
            })
        current += datetime.timedelta(days=freq_days)

    # Deduplicate: if two consecutive checkpoints map to same revid, keep both
    # (the composition didn't change, but we still record the week)
    return checkpoints

In [14]:
def fetch_revision_html(revid: int) -> str:
    """Fetch the rendered HTML of a specific Wikipedia revision."""
    params = {
        "title":  ARTICLE,
        "oldid":  revid,
        "action": "render",   # returns just the article body HTML
    }
    resp = SESSION.get(WIKI_OLD, params=params, timeout=30)
    resp.raise_for_status()
    return resp.text

In [15]:
def extract_components(html: str) -> list[dict]:
    """
    Parse the S&P 100 components table from the article HTML.
    Returns a list of dicts: {ticker, company}

    The Wikipedia article has had slightly different table structures over time.
    We try multiple heuristics in order of reliability.
    """
    soup = BeautifulSoup(html, "html.parser")
    results = []

    # --- Strategy 1: find wikitable with a "Symbol" or "Ticker" header column ---
    for table in soup.find_all("table", class_=re.compile(r"wikitable")):
        headers = [th.get_text(strip=True).lower()
                   for th in table.find_all("th")]
        # Look for ticker-like and company-like columns
        ticker_col = next((i for i, h in enumerate(headers)
                           if h in ("symbol", "ticker", "tick.")), None)
        name_col   = next((i for i, h in enumerate(headers)
                           if h in ("company", "name", "security")), None)

        if ticker_col is None:
            continue  # not the right table

        rows = table.find_all("tr")[1:]  # skip header row
        for row in rows:
            cells = row.find_all(["td", "th"])
            if not cells or len(cells) <= max(
                    ticker_col, name_col if name_col else 0):
                continue
            ticker  = cells[ticker_col].get_text(strip=True).upper()
            company = (cells[name_col].get_text(strip=True)
                       if name_col is not None and name_col < len(cells)
                       else "")
            # Basic sanity: tickers are 1-5 uppercase letters
            if re.match(r"^[A-Z]{1,5}$", ticker):
                results.append({"ticker": ticker, "company": company})

        if results:
            return results   # found a good table, stop searching

    # --- Strategy 2: fallback — look for any table with ≥80 rows (likely the
    #     full component list even if headers differ) ---
    for table in soup.find_all("table"):
        rows = table.find_all("tr")
        if len(rows) < 80:
            continue
        for row in rows[1:]:
            cells = row.find_all("td")
            if not cells:
                continue
            # First cell is often the ticker in older layouts
            candidate = cells[0].get_text(strip=True).upper()
            if re.match(r"^[A-Z]{1,5}$", candidate):
                company = cells[1].get_text(strip=True) if len(cells) > 1 else ""
                results.append({"ticker": candidate, "company": company})
        if len(results) >= 80:
            return results

    return results  # may be empty if parsing failed

In [16]:
def build_composition_records(checkpoints: list[dict],
                               cache_dir: str) -> pd.DataFrame:
    """
    For each checkpoint, fetch the revision HTML, extract components,
    and return a long-format DataFrame.
    Caches raw HTML to disk to allow re-runs without re-fetching.
    """
    os.makedirs(cache_dir, exist_ok=True)
    records = []
    failed  = []

    for cp in tqdm(checkpoints, desc="Fetching revisions"):
        revid      = cp["revid"]
        checkpoint = cp["checkpoint"]
        cache_path = os.path.join(cache_dir, f"rev_{revid}.html")

        # Load from cache or fetch
        if os.path.exists(cache_path):
            with open(cache_path, "r", encoding="utf-8") as f:
                html = f.read()
        else:
            try:
                html = fetch_revision_html(revid)
                with open(cache_path, "w", encoding="utf-8") as f:
                    f.write(html)
                time.sleep(SLEEP_BETWEEN_REQUESTS)
            except Exception as e:
                print(f"\n  ⚠ Failed to fetch revid {revid} ({checkpoint}): {e}")
                failed.append(cp)
                continue

        components = extract_components(html)
        if not components:
            print(f"\n  ⚠ No components parsed for revid {revid} ({checkpoint})")
            failed.append(cp)
            continue

        for comp in components:
            records.append({
                "checkpoint": checkpoint,
                "revid":      revid,
                "ticker":     comp["ticker"],
                "company":    comp["company"],
            })

    if failed:
        print(f"\n  {len(failed)} checkpoints failed — see logs above.")

    return pd.DataFrame(records)

In [17]:
def classify_companies(long_df: pd.DataFrame,
                        total_checkpoints: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Given the long-format composition DataFrame, classify each ticker
    as CONSTANT (present every week) or TRANSIENT (absent at least once).

    Returns (constant_df, transient_df).
    """
    stats = (
        long_df.groupby("ticker")
        .agg(
            company         = ("company", "last"),  # most recent name
            weeks_present   = ("checkpoint", "nunique"),
            first_seen      = ("checkpoint", "min"),
            last_seen       = ("checkpoint", "max"),
        )
        .reset_index()
    )
    stats["presence_pct"] = (stats["weeks_present"] / total_checkpoints * 100).round(2)
    stats["status"] = stats["weeks_present"].apply(
        lambda w: "CONSTANT" if w == total_checkpoints else "TRANSIENT"
    )

    constant  = stats[stats["status"] == "CONSTANT"].sort_values("ticker")
    transient = (stats[stats["status"] == "TRANSIENT"]
                 .sort_values("presence_pct", ascending=False))

    return constant, transient

In [18]:
def write_summary(constant: pd.DataFrame,
                  transient: pd.DataFrame,
                  total_checkpoints: int,
                  start: datetime.date,
                  end: datetime.date,
                  path: str) -> None:
    lines = [
        "=" * 60,
        "  S&P 100 COMPOSITION ANALYSIS",
        f"  Period  : {start} → {end}",
        f"  Snapshots sampled : {total_checkpoints}",
        "=" * 60,
        "",
        f"CONSTANT companies (present in all {total_checkpoints} snapshots): "
        f"{len(constant)}",
        "-" * 40,
    ]
    for _, row in constant.iterrows():
        lines.append(f"  {row['ticker']:<6}  {row['company']}")

    lines += [
        "",
        f"TRANSIENT companies (absent in ≥1 snapshot): {len(transient)}",
        "-" * 40,
        f"  {'Ticker':<6}  {'Presence%':>10}  {'Weeks':>6}  "
        f"{'First seen':<12}  {'Last seen':<12}  Company",
    ]
    for _, row in transient.iterrows():
        lines.append(
            f"  {row['ticker']:<6}  {row['presence_pct']:>9.1f}%  "
            f"{int(row['weeks_present']):>6}  "
            f"{str(row['first_seen']):<12}  {str(row['last_seen']):<12}  "
            f"{row['company']}"
        )

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("\n".join(lines))

In [19]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cache_dir = os.path.join(OUTPUT_DIR, "_html_cache")

    print(f"[1/5] Fetching revision index for S&P_100 "
          f"({START_DATE} → {END_DATE})…")
    revisions = get_revision_ids(START_DATE, END_DATE)
    print(f"      {len(revisions)} total revisions found.")

    print(f"\n[2/5] Sampling one revision per {SAMPLE_FREQ_DAYS} days…")
    checkpoints = sample_weekly_revisions(
        revisions, START_DATE, END_DATE, SAMPLE_FREQ_DAYS)
    print(f"      {len(checkpoints)} weekly checkpoints selected.")

    # Save the revision index
    rev_df = pd.DataFrame(checkpoints)
    rev_df.to_csv(os.path.join(OUTPUT_DIR, "revisions_index.csv"), index=False)

    print(f"\n[3/5] Fetching & parsing HTML for each checkpoint "
          f"(cached in {cache_dir})…")
    long_df = build_composition_records(checkpoints, cache_dir)

    if long_df.empty:
        print("ERROR: No composition data could be extracted. "
              "Check parsing or network access.")
        return

    long_df.to_csv(
        os.path.join(OUTPUT_DIR, "weekly_composition.csv"), index=False)
    print(f"      {len(long_df)} total (checkpoint × ticker) records saved.")

    print("\n[4/5] Building composition matrix…")
    matrix = (long_df
              .assign(present=1)
              .pivot_table(index="checkpoint", columns="ticker",
                           values="present", fill_value=0))
    matrix.to_csv(os.path.join(OUTPUT_DIR, "composition_matrix.csv"))

    print("\n[5/5] Classifying companies…")
    n_checkpoints = len(checkpoints)
    constant, transient = classify_companies(long_df, n_checkpoints)

    constant.to_csv(
        os.path.join(OUTPUT_DIR, "constant_companies.csv"), index=False)
    transient.to_csv(
        os.path.join(OUTPUT_DIR, "transient_companies.csv"), index=False)

    write_summary(
        constant, transient, n_checkpoints,
        START_DATE, END_DATE,
        os.path.join(OUTPUT_DIR, "summary_report.txt"),
    )

    print(f"\n✅ All outputs written to: {OUTPUT_DIR}/")

In [20]:
main()

[1/5] Fetching revision index for S&P_100 (2021-01-01 → 2025-12-31)…
      111 total revisions found.

[2/5] Sampling one revision per 7 days…
      258 weekly checkpoints selected.

[3/5] Fetching & parsing HTML for each checkpoint (cached in ./sp100_output\_html_cache)…


Fetching revisions: 100%|██████████| 258/258 [02:09<00:00,  1.98it/s]


      25800 total (checkpoint × ticker) records saved.

[4/5] Building composition matrix…

[5/5] Classifying companies…
  S&P 100 COMPOSITION ANALYSIS
  Period  : 2021-01-01 → 2025-12-31
  Snapshots sampled : 258

CONSTANT companies (present in all 258 snapshots): 88
----------------------------------------
  AAPL    Apple Inc.
  ABBV    AbbVie
  ABT     Abbott Laboratories
  ACN     Accenture
  ADBE    Adobe Inc.
  AIG     American International Group
  AMGN    Amgen
  AMT     American Tower
  AMZN    Amazon
  AXP     American Express
  BA      Boeing
  BAC     Bank of America
  BK      BNY Mellon
  BKNG    Booking Holdings
  BLK     BlackRock
  BMY     Bristol Myers Squibb
  C       Citigroup
  CAT     Caterpillar Inc.
  CL      Colgate-Palmolive
  CMCSA   Comcast
  COF     Capital One
  COP     ConocoPhillips
  COST    Costco
  CRM     Salesforce
  CSCO    Cisco
  CVS     CVS Health
  CVX     Chevron Corporation
  DHR     Danaher Corporation
  DIS     Walt Disney Company (The)
  DU